# 03 | Feature engineering: 12 selected features

Time-window aggregation, listening trends, subscriptions, and selection of 12 predictive features.

**Source:** `archive/Project_fin_original.ipynb`. The original computations, structure, and results have been preserved. The notebook text has been translated into English; standalone setup and local checkpoints support reproducible execution.


In [ ]:
# Standalone setup: no src/ folder required.
# Run the notebooks in numerical order (02 -> 07); 01 is optional exploration.
from pathlib import Path
import os
import sys
import numpy as np
import pandas as pd

working_dir = Path.cwd().resolve()
# Works when VS Code/Jupyter starts in the repository root or notebooks/.
if working_dir.name == "notebooks":
    project_root = working_dir.parent
elif (working_dir / "notebooks").is_dir():
    project_root = working_dir
elif (working_dir.parent / "notebooks").is_dir():
    project_root = working_dir.parent
else:
    project_root = working_dir

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

HORIZON = pd.Timedelta(days=10)
CUTOFFS = pd.date_range(start="2018-10-11", end="2018-11-10", freq="10D")
FEATURES_12 = [
    "n_sessions_10d", "n_active_days_10d", "n_songs_10d",
    "n_likes_10d", "n_dislikes_10d", "n_errors_10d",
    "n_playlist_adds_10d", "total_song_length_10d", "n_artists_10d",
    "downgraded_10d", "is_paid", "songs_trend_10d"
]
EVENT_COLS = [
    "userId", "time", "registration", "page", "sessionId", "level",
    "gender", "length", "artist", "song", "status", "method",
    "userAgent", "auth", "firstName", "lastName", "itemInSession", "location"
]

def raw_path(kind):
    if kind not in ("train", "test"):
        raise ValueError("kind must be 'train' or 'test'")
    filename = f"{kind}.parquet"
    candidates = [
        project_root / filename,
        project_root / "data" / "raw" / filename,
        project_root / "notebooks" / filename,
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(
        f"Missing {filename}. Put it in the repository root or data/raw/. "
        f"Current root: {project_root}"
    )

def load_events(kind):
    frame = pd.read_parquet(raw_path(kind), columns=EVENT_COLS)
    frame["userId"] = frame["userId"].astype(str)
    frame["time"] = pd.to_datetime(frame["time"])
    frame["registration"] = pd.to_datetime(frame["registration"])
    return frame

def data_path(filename):
    folder = project_root / "data" / "processed"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def artifact_path(filename):
    folder = project_root / "artifacts"
    folder.mkdir(parents=True, exist_ok=True)
    return folder / filename

def save_cv_state(dev_data, holdout_data, cv_splits):
    dev_data.to_parquet(data_path("dev_data.parquet"), index=False)
    holdout_data.to_parquet(data_path("holdout_data.parquet"), index=False)
    fold_arrays = {}
    for i, (idx_train, idx_valid) in enumerate(cv_splits):
        fold_arrays[f"train_{i}"] = np.asarray(idx_train, dtype=np.int64)
        fold_arrays[f"valid_{i}"] = np.asarray(idx_valid, dtype=np.int64)
    np.savez_compressed(data_path("cv_splits.npz"), **fold_arrays)

def load_cv_state():
    dev = pd.read_parquet(data_path("dev_data.parquet"))
    holdout = pd.read_parquet(data_path("holdout_data.parquet"))
    with np.load(data_path("cv_splits.npz")) as saved:
        n = sum(name.startswith("train_") for name in saved.files)
        splits = [
            (saved[f"train_{i}"].copy(), saved[f"valid_{i}"].copy())
            for i in range(n)
        ]
    return dev, holdout, splits

print("Project folder:", project_root)

train = load_events("train")
labels_all = pd.read_parquet(data_path("labels_all.parquet"))


# 3) Feature engineering

In [19]:
all_datasets = []

data_start = train["time"].min().floor("D")

for cutoff in CUTOFFS:

    history_start = cutoff - HORIZON

    # 1. EVENTS FROM THE LAST 10 DAYS
    history = train.loc[
        (train["time"] >= history_start) &
        (train["time"] < cutoff),
        [
            "userId", "time", "page", "sessionId",
            "length", "artist", "song"
        ]
    ].copy()

    # 2. EVENT-LEVEL INDICATORS
    history["day"] = history["time"].dt.floor("D")
    history["is_song"] = history["page"].eq("NextSong")
    history["is_like"] = history["page"].eq("Thumbs Up")
    history["is_dislike"] = history["page"].eq("Thumbs Down")
    history["is_error"] = history["page"].eq("Error")
    history["is_playlist"] = history["page"].eq("Add to Playlist")
    history["is_downgrade"] = history["page"].eq("Submit Downgrade")

    # 3. ACTIVITY, ENGAGEMENT, AND MUSIC FEATURES
    features_period = history.groupby("userId").agg(
        n_events_10d=("page", "size"),
        n_sessions_10d=("sessionId", "nunique"),
        n_active_days_10d=("day", "nunique"),
        n_songs_10d=("is_song", "sum"),
        n_likes_10d=("is_like", "sum"),
        n_dislikes_10d=("is_dislike", "sum"),
        n_errors_10d=("is_error", "sum"),
        n_playlist_adds_10d=("is_playlist", "sum"),
        total_song_length_10d=("length", "sum"),
        n_artists_10d=("artist", "nunique"),
        n_unique_songs_10d=("song", "nunique"),

        # New feature: subscription downgrade
        downgraded_10d=("is_downgrade", "max")
    ).reset_index()

    # 4. GET TARGET LABELS FOR THIS WINDOW
    labels_period = labels_all.loc[
        labels_all["cutoff"] == cutoff
    ].copy()

    # 5. JOIN TARGET LABELS AND FEATURES
    dataset_period = labels_period.merge(
        features_period,
        on="userId",
        how="left",
        validate="one_to_one"
    )

   ### Users with no activity during the last 10 days
    count_cols = features_period.columns.drop("userId")

    dataset_period[count_cols] = (
        dataset_period[count_cols].fillna(0)
    )

    # 6. CURRENT SUBSCRIPTION FEATURE
    past = train.loc[
        train["time"] < cutoff,
        ["userId", "time", "level"]
    ]

    ## Last recorded event for each user
    last_idx = past.groupby("userId")["time"].idxmax()

    last_levels = (
        past.loc[last_idx]
        .set_index("userId")["level"]
    )

    # Paid = 1, free = 0
    dataset_period["is_paid"] = (
        dataset_period["userId"]
        .map(last_levels)
        .map({"paid": 1, "free": 0})
    )

    # 7. LISTENING ACTIVITY TREND
    previous_start = cutoff - 2 * HORIZON
    previous_end = cutoff - HORIZON

    if previous_start >= data_start:

        # Song events during the previous 10-day window
        previous_songs = train.loc[
            (train["time"] >= previous_start) &
            (train["time"] < previous_end) &
            (train["page"] == "NextSong"),
            "userId"
        ].value_counts()

        n_previous = (
            dataset_period["userId"]
            .map(previous_songs)
            .fillna(0)
        )

        n_recent = dataset_period["n_songs_10d"]

        # Normalized change between -1 and +1
        dataset_period["songs_trend_10d"] = (
            (n_recent - n_previous) /
            (n_recent + n_previous + 1)
        )

    else:
        # Prior history is unavailable
        dataset_period["songs_trend_10d"] = np.nan

    
    all_datasets.append(dataset_period)


dataset_all = pd.concat(
    all_datasets,
    ignore_index=True
)

### Select 12 model features

FEATURES_12 = [
    "n_sessions_10d",
    "n_active_days_10d",
    "n_songs_10d",
    "n_likes_10d",
    "n_dislikes_10d",
    "n_errors_10d",
    "n_playlist_adds_10d",
    "total_song_length_10d",
    "n_artists_10d",
    "downgraded_10d",
    "is_paid",
    "songs_trend_10d"
]

dataset_model = dataset_all[
    ["userId", "cutoff", "churn"] + FEATURES_12
].copy()

### Sanity checks

print("Complete dataset:", dataset_all.shape)
print("Dataset with 12 features:", dataset_model.shape)

print("\nObservations by prediction date:")
print(dataset_model.groupby("cutoff").size())

print("\nMissing values:")
print(dataset_model[FEATURES_12].isna().sum())

display(dataset_model.head())

Complete dataset: (60434, 17)
Dataset with 12 features: (60434, 15)

Observations by prediction date:
cutoff
2018-10-11    14176
2018-10-21    15452
2018-10-31    15530
2018-11-10    15276
dtype: int64

Missing values:
n_sessions_10d               0
n_active_days_10d            0
n_songs_10d                  0
n_likes_10d                  0
n_dislikes_10d               0
n_errors_10d                 0
n_playlist_adds_10d          0
total_song_length_10d        0
n_artists_10d                0
downgraded_10d               0
is_paid                      0
songs_trend_10d          14176
dtype: int64


,userId,cutoff,churn,n_sessions_10d,n_active_days_10d,n_songs_10d,n_likes_10d,n_dislikes_10d,n_errors_10d,n_playlist_adds_10d,total_song_length_10d,n_artists_10d,downgraded_10d,is_paid,songs_trend_10d
0,1749042,2018-10-11,0,6.0,8.0,486.0,21.0,7.0,0.0,13.0,121710.35590,416.0,False,1,NaN
1,1563081,2018-10-11,0,1.0,1.0,25.0,1.0,0.0,0.0,0.0,6304.72010,23.0,False,1,NaN
2,1697168,2018-10-11,0,9.0,9.0,339.0,15.0,5.0,1.0,8.0,84159.43194,305.0,True,0,NaN
3,1222580,2018-10-11,0,11.0,9.0,1275.0,59.0,12.0,2.0,37.0,320254.37710,957.0,True,1,NaN
4,1714398,2018-10-11,0,4.0,4.0,327.0,19.0,4.0,1.0,10.0,84535.59698,293.0,False,1,NaN


In [ ]:
# Save checkpoints for the complete dataset and the 12-feature dataset
dataset_all.to_parquet(data_path("dataset_all.parquet"), index=False)
dataset_model.to_parquet(data_path("dataset_model.parquet"), index=False)
print("Prepared datasets saved.")